# Integrate planning and daily close

A live application plans before a demand epoch and closes that epoch when its
realized demand becomes available. Stockcast provides the state, schedule and
policy primitives; the application owns persistence and external ordering.

This notebook replays four historical dates to demonstrate both phases. The
`close_batch` wrapper is an offline convenience, not a claim that a decision
computed after observing a day was actually submitted before that day.


## The online boundary

```text
previous closed state + history through its date
    -> open_and_plan(next_date): receive, refit if scheduled, predict, accept order
    -> persist planned state and order evidence
    -> current demand becomes available
    -> complete_close(batch): fulfill demand, append observed history
    -> persist completed state
```

Current demand is not an input to `open_and_plan`. A weekly file can be replayed
one date at a time, but replay orders must never be represented as historical
live submissions. Both planning and close require application-owned atomic
persistence and idempotency.


## 1. Load the components

`InventoryStateDataFrame` owns the live stock and pipeline state. `OrderUpToPolicy` reads that state and proposes replenishment. `update_inventory_with_orders` schedules an accepted decision with immediate receipt for zero lead time, or future pipeline for positive lead time.

Smooth represents an external forecast service. Forecast fitting is deliberately outside Stockcast.


In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm
from smooth import ES
from stockcast import InventoryStateDataFrame, OrderUpToPolicy
from stockcast.utils import update_inventory_with_orders


In [3]:
LEAD_TIME = 2
REVIEW_PERIOD = 2
PROTECTION_HORIZON = LEAD_TIME + REVIEW_PERIOD
SERVICE_LEVEL = 0.95
FREQUENCY = 'D'


## 2. Restore yesterday's committed state

Production starts from an observed or previously committed state—not from a demand-based inventory guess. We use an explicit historical February 28 checkpoint. It contains all 100 SKUs, their on-hand balances, and both lead-time pipeline slots.

This is a saved historical opening state, not output from the current Notebook 09 run; the replay below uses the before-demand decision sequence shown here.

The few conversion lines below are storage-adapter work. A real repository would perform the equivalent schema and checksum validation before returning these objects.


In [4]:
checkpoint_path = Path(
    'examples/notebooks/data/m5/notebook08_covariance_normal_checkpoint_2016-02-28.json'
)
checkpoint = json.loads(checkpoint_path.read_text())

# Storage rows become the package's typed live-state object.
inventory_rows = pd.DataFrame(checkpoint['inventory_rows'])
inventory_rows['date'] = pd.to_datetime(inventory_rows['date'])
inventory_rows['in_transit'] = inventory_rows['in_transit'].map(
    lambda slots: np.asarray(slots, dtype=float)
)

inventory = InventoryStateDataFrame(
    inventory_rows, max_lead_time=LEAD_TIME, allow_backorders=False
)

# Restored live state: 100 SKUs with on-hand balances and pipeline slots.
inventory

InventoryStateDataFrame(n_skus=100, total_on_hand=1205, total_safety_stock=0, total_backorders=0, has_stockout=False, has_backorder=False)

In [5]:
saved_policy = checkpoint['active_policy']
# Restore the policy that was active when yesterday's state was committed.
saved_targets = pd.DataFrame(saved_policy['target_rows'])
saved_targets['target_end_date'] = pd.to_datetime(saved_targets['target_end_date'])

active_policy = OrderUpToPolicy(
    lead_time=LEAD_TIME, review_period=REVIEW_PERIOD,
    service_level=SERVICE_LEVEL, allow_backorders=False,
).fit(
    saved_targets[['unique_id', 'cumulative_target', 'target_end_date']],
    forecast_origin=saved_policy['target_metadata']['forecast_origin'],
    forecast_frequency=FREQUENCY, target_column='cumulative_target',
    target_end_date_column='target_end_date', target_probability=SERVICE_LEVEL,
    protection_horizon=PROTECTION_HORIZON, target_source='external_direct',
)

# Restored targets carried over from yesterday's committed policy.
active_policy.get_target_levels().head()

,unique_id,target_level
0,FOODS_1_001_CA_1,7.318559
1,FOODS_1_002_CA_1,4.273597
2,FOODS_1_003_CA_1,7.258375
3,FOODS_1_004_CA_1,26.925876
4,FOODS_1_005_CA_1,11.901558


In [6]:
opening = inventory.get_dataframe()
# A small preview is enough to confirm date, period, stock, and pipeline shape.
preview = opening.head(3).assign(
    in_transit=lambda frame: frame.in_transit.map(list)
)
print(preview[['unique_id', 'date', 'period', 'on_hand', 'in_transit']].to_string(index=False))


       unique_id       date  period  on_hand in_transit
FOODS_1_001_CA_1 2016-02-28    28.0      9.8 [0.0, 0.0]
FOODS_1_002_CA_1 2016-02-28    28.0      6.0 [0.0, 0.0]
FOODS_1_003_CA_1 2016-02-28    28.0      4.9 [0.0, 0.0]


## 3. Treat a weekly delivery as dated daily batches

The example file contains several dates together because it is an offline dataset. A live system must release only observations that actually exist by the close date. Otherwise a review on Tuesday could learn from Wednesday's demand.

History therefore ends at the checkpoint date, and the walkthrough releases four subsequent slices in chronological order. A real daily feed would usually supply one completed slice at a time.


In [7]:
demand = pd.read_csv('examples/notebooks/data/m5/sample_1.csv', parse_dates=['date'])
demand = demand.sort_values(['date', 'unique_id']).reset_index(drop=True)

# Forecast history stops exactly where the opening checkpoint stops.
history = demand[demand.date.le('2016-02-28')][['unique_id', 'date', 'y']].copy()
weekly_delivery = demand[demand.date.between('2016-02-29', '2016-03-06')][
    ['unique_id', 'date', 'y']
].copy()

def release_batch(delivery, close_date):
    # Future dates remain in the landing table until their scheduled close.
    return delivery[delivery.date.eq(pd.Timestamp(close_date))].copy()

# Seven releasable dates times one hundred SKUs land together.
assert weekly_delivery.groupby('date').size().eq(100).all()
assert history.date.max() == pd.Timestamp('2016-02-28')
assert weekly_delivery.date.min() == pd.Timestamp('2016-02-29')

print(f'Landing rows: {len(weekly_delivery):,} = 7 dates × 100 SKUs')


Landing rows: 700 = 7 dates × 100 SKUs


## 4. Keep policy fitting outside the state transition

On a review date, the forecasting service receives observations through the preceding closed date and returns one cumulative four-day target per SKU. This is the same covariance-normal construction used earlier; it is included only so the production handoff remains executable.

The important interface is the return value: a fitted, dated Stockcast policy. `process_demand` itself never fits a forecasting model.


In [8]:
def forecast_target(sku_history, origin):
    series = sku_history.sort_values('date').y.astype(float).reset_index(drop=True)
    model = ES(model='ANN').fit(series)  # One independent model per SKU.
    forecast = model.predict(
        h=PROTECTION_HORIZON, interval='approximate',
        level=SERVICE_LEVEL, side='upper', cumulative=True,
    )
    errors = model.rmultistep(h=PROTECTION_HORIZON).astype(float)
    # Joint error paths preserve dependence across the four horizons.
    sigma = np.cov(errors.to_numpy(), rowvar=False, ddof=1)
    ones = np.ones(PROTECTION_HORIZON)
    target = float(forecast.mean.iloc[-1]) + norm.ppf(SERVICE_LEVEL) * np.sqrt(ones @ sigma @ ones)
    return {'unique_id': sku_history.unique_id.iloc[0], 'target': target,
            'target_end_date': origin + pd.Timedelta(days=PROTECTION_HORIZON)}


In [9]:
def fit_review_policy(observations, origin):
    origin = pd.Timestamp(origin)
    target_rows = [
        forecast_target(sku_history, origin)
        for _, sku_history in observations.groupby('unique_id', sort=True)
    ]
    targets = pd.DataFrame(target_rows)
    # Stockcast receives direct cumulative targets, not the forecast model.
    policy = OrderUpToPolicy(
        lead_time=LEAD_TIME, review_period=REVIEW_PERIOD,
        service_level=SERVICE_LEVEL, allow_backorders=False,
    ).fit(
        targets, forecast_origin=origin, forecast_frequency=FREQUENCY,
        target_column='target', target_end_date_column='target_end_date',
        target_probability=SERVICE_LEVEL, protection_horizon=PROTECTION_HORIZON,
        target_source='external_direct',
    )
    return policy, targets


## 5. Separate planning from closing

`open_and_plan` uses only history through the prior closed date. It receives due
stock and makes an eligible decision before current demand. `complete_close`
then validates and fulfills the current batch and appends it to observed history.
The `close_batch` wrapper below combines them only for this historical replay.

The functions return new objects. A live adapter commits each phase separately,
with its own date/phase idempotency key and order evidence.


In [10]:
def open_and_plan(inventory, policy, history, next_date):
    # Run at the start of the demand epoch. No current demand is an input.
    information_date = pd.Timestamp(inventory.data.date.iloc[0])
    if history.date.max() > information_date:
        raise ValueError('forecast history extends beyond the pre-demand cutoff')
    if pd.Timestamp(next_date) != information_date + pd.Timedelta(days=1):
        raise ValueError('the planning date must be the next daily epoch')
    next_period = int(inventory.data.period.iloc[0]) + 1
    # Explicit caller-owned phase continuing this historical checkpoint calendar.
    is_review = next_period % REVIEW_PERIOD == 0
    opened = inventory.advance_period(period_frequency=FREQUENCY, is_review_period=is_review)
    decision, targets = None, None
    if is_review:
        policy, targets = fit_review_policy(history, information_date)
        decision = policy.predict(opened, current_period=next_period)
        opened = update_inventory_with_orders(opened, decision, policy=policy)
    return opened, policy, decision, targets


def complete_close(opened, history, batch):
    # Run after this epoch's demand arrives. No new order is inferred here.
    closed = opened.fulfill_demand(batch)
    next_history = pd.concat([history, batch], ignore_index=True)
    return closed, next_history


def close_batch(inventory, policy, history, batch):
    # Historical demonstration wrapper only: replay both phases in one call.
    # Validate a defensive scratch transition before the external forecast work.
    import copy
    copy.deepcopy(inventory).process_demand(batch, review_period=policy.review_period, period_frequency=FREQUENCY)
    opened, policy, decision, targets = open_and_plan(inventory, policy, history, batch.date.iloc[0])
    closed, next_history = complete_close(opened, history, batch)
    return closed, policy, next_history, decision, targets


## 6. Non-review close: February 29

The checkpoint is at period 28. February 29 advances it to period 29, which is not divisible by the two-day review period. Receipts, pipeline progression, demand fulfillment, shortages, and the state clock are still processed. Only forecasting and new order placement are skipped.


In [11]:
feb29 = release_batch(weekly_delivery, '2016-02-29')
# The same close function handles both review and non-review dates.
feb29_opening = inventory.get_dataframe()
inventory, active_policy, history, feb29_order, _ = close_batch(
    inventory, active_policy, history, feb29
)

feb29_state = inventory.get_dataframe()
feb29_flow = pd.Series({
    'starting_on_hand': feb29_opening.on_hand.sum(),
    'received': feb29_state.latest_received.sum(),
    'demand': feb29_state.latest_incoming_demand.sum(),
    'fulfilled': feb29_state.latest_fulfilled.sum(),
    'shortage': feb29_state.latest_shortage.sum(),
    'ending_on_hand': feb29_state.on_hand.sum(),
})
print(feb29_flow.to_string())
print('\nPeriod:', int(feb29_state.period.iloc[0]),
      '| Review:', bool(feb29_state.is_review_period.iloc[0]),
      '| New order:', feb29_order)
print('Stock balance holds:', np.isclose(
    feb29_flow.starting_on_hand + feb29_flow.received - feb29_flow.fulfilled,
    feb29_flow.ending_on_hand,
))


starting_on_hand    1205.2
received               0.0
demand               148.0
fulfilled            121.0
shortage              27.0
ending_on_hand      1084.2

Period: 29 | Review: False | New order: None
Stock balance holds: True


## 7. Review epoch: March 1

March 1 is state period 30, so the explicit two-day calendar enables planning.
The forecast origin is **February 29**: March 1 demand is excluded. The order
is placed before March 1 demand and arrives before March 3 demand. The replay
then closes March 1 and appends its observations.


In [12]:
mar01 = release_batch(weekly_delivery, '2016-03-01')
# Period 30 takes the review branch inside close_batch.
inventory, active_policy, history, mar01_order, mar01_targets = close_batch(
    inventory, active_policy, history, mar01
)

mar01_state = inventory.get_dataframe()
print('Review:', bool(mar01_state.is_review_period.iloc[0]))
print('Forecast origin:', active_policy.get_target_metadata()['forecast_origin'][:10])
print('Fitted SKUs:', len(mar01_targets))


Review: True
Forecast origin: 2016-02-29
Fitted SKUs: 100


In [13]:
order_preview = mar01_order.get_dataframe().nlargest(5, 'order_quantity')[
    ['unique_id', 'order_quantity', 'order_period', 'expected_delivery_period']
]
print(order_preview.to_string(index=False))


       unique_id  order_quantity  order_period  expected_delivery_period
FOODS_1_099_CA_1       66.627819            30                        32
FOODS_1_085_CA_1       53.969760            30                        32
FOODS_1_019_CA_1       39.313250            30                        32
FOODS_1_018_CA_1       37.431184            30                        32
FOODS_1_082_CA_1       34.993023            30                        32


## 8. Follow the order through the pipeline

March 2 is non-review, so the March 1 order moves one slot closer without a new decision. On March 3 it is received before demand; period 32 then opens the next review.

The table connects the March 1 `OrderDecision` to the March 3 state flow fields. This is caller-owned operational evidence, not the canonical event ledger returned by `SimulationEngine`.


In [14]:
daily_log = []
# Process each date only when the scheduler releases it.
for close_date in ['2016-03-02', '2016-03-03']:
    batch = release_batch(weekly_delivery, close_date)
    inventory, active_policy, history, decision, _ = close_batch(
        inventory, active_policy, history, batch
    )
    state = inventory.get_dataframe()
    daily_log.append({
        'date': close_date, 'period': int(state.period.iloc[0]),
        'review': bool(state.is_review_period.iloc[0]),
        'received': state.latest_received.sum(),
        'ordered': state.latest_order.sum(),
        'pipeline': state.in_transit.map(np.sum).sum(),
        'ending_on_hand': state.on_hand.sum(),
    })

print(pd.DataFrame(daily_log).to_string(index=False))


      date  period  review  received    ordered   pipeline  ending_on_hand
2016-03-02      31   False   0.00000   0.000000 566.447040       926.80000
2016-03-03      32    True 566.44704 116.516611 116.516611      1361.24704


In [15]:
placed = mar01_order.get_dataframe()[['unique_id', 'order_quantity']]
# March 3 latest_received is the quantity removed from pipeline slot 0.
received = inventory.get_dataframe()[['unique_id', 'latest_received']]
receipt_check = placed.merge(received, on='unique_id', validate='one_to_one')

print(receipt_check.nlargest(5, 'order_quantity').to_string(index=False))
print('All March 1 orders received on March 3:',
      np.allclose(receipt_check.order_quantity, receipt_check.latest_received))


       unique_id  order_quantity  latest_received
FOODS_1_099_CA_1       66.627819        66.627819
FOODS_1_085_CA_1       53.969760        53.969760
FOODS_1_019_CA_1       39.313250        39.313250
FOODS_1_018_CA_1       37.431184        37.431184
FOODS_1_082_CA_1       34.993023        34.993023
All March 1 orders received on March 3: True


## 9. Connect the two phases to a repository

A live scheduler calls `open_and_plan` at the beginning of the next demand epoch,
then persists its planned state, policy and order evidence atomically. After
observations arrive, it calls `complete_close` on that planned state and commits
the completed inventory plus history. Each phase needs an idempotency key so a
retry does not create another order or fulfill demand twice.

A file containing several past days supports a historical replay through
`close_batch`. It does not prove that orders were submitted on those past dates.
An external application must explicitly distinguish replay from live ordering.


## Which Stockcast path should production use?

Use the component path shown here when the application deliberately owns online orchestration:

```text
advance_period -> schedule -> policy.predict -> update_inventory_with_orders -> fulfill_demand
```

Use `SimulationEngine` for backtests, multi-period replay, comparisons, or when you require its complete preflight, callbacks, ordered constraints, canonical events, audits, evaluation windows, and run manifest.

The current component path does **not** recreate those engine services. A production system that requires them needs either one-period engine runs—with careful run-reset semantics—or a future package-level single-period API. This notebook does not invent that missing API or claim that caller snapshots are canonical events.
